In [1]:
import os
import sys
from pathlib import Path
from typing import Any

sys.path.append(os.path.abspath(".."))

import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm


import torch
import lightning as L
from lightning.fabric.utilities import measure_flops
from lightning.pytorch.utilities.model_summary.model_summary import ModelSummary
from timm.models.resnet import resnet18, resnet34, resnet50, resnet101, resnet152
from timm.models.convnext import (
    convnext_small,
    convnext_tiny,
)
from timm.models.swin_transformer import (
    swin_tiny_patch4_window7_224,
    swin_small_patch4_window7_224,
)
from timm.models.vision_transformer import vit_base_patch16_224
from timm.models.mobilenetv3 import (
    mobilenetv4_conv_medium,
    mobilenetv4_conv_small,
    mobilenetv4_hybrid_medium,
    mobilenetv4_conv_large,
    mobilenetv4_hybrid_large,
)
from timm.models.mobilevit import (
    mobilevitv2_050,
    mobilevitv2_075,
    mobilevitv2_100,
    mobilevitv2_125,
    mobilevitv2_150,
    mobilevitv2_175,
    mobilevitv2_200,
)

plt.style.use("default")
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.family"] = ["serif"]
plt.rcParams["font.serif"] = ["Times New Roman"]

SEEDS = [0, 21, 42, 84, 168, 336, 672, 1344, 2688, 3407]

BASE_DIR = Path(os.path.abspath("."))
PROJ_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "data"
if not DATA_DIR.exists():
    DATA_DIR.mkdir(parents=True)

LOGS_DIR = PROJ_DIR / "logs" / "repeat"
REPORT_DIR = DATA_DIR / "report"

DATASET_DIR = Path("/dev/shm/data/dataset/seeds_rgb")
path_train = DATASET_DIR / "cls_train_656_rgb.json"
path_val = DATASET_DIR / "cls_val_656_rgb.json"
path_test = DATASET_DIR / "cls_test_656_rgb.json"
BATCH_SIZE = 256

In [2]:
ARCHS = {
    "resnet_18": resnet18,
    "resnet_34": resnet34,
    "resnet_50": resnet50,
    "resnet_101": resnet101,
    "resnet_152": resnet152,
    "swin_tiny": swin_tiny_patch4_window7_224,
    "swin_small": swin_small_patch4_window7_224,
    # "swin_base": swin_base_patch4_window7_224,
    "convnext_tiny": convnext_tiny,
    "convnext_small": convnext_small,
    # "convnext_base": convnext_base,
    "vit_b": vit_base_patch16_224,
    "mobilevit_050": mobilevitv2_050,
    "mobilevit_075": mobilevitv2_075,
    "mobilevit_100": mobilevitv2_100,
    "mobilevit_125": mobilevitv2_125,
    "mobilevit_150": mobilevitv2_150,
    "mobilevit_175": mobilevitv2_175,
    "mobilevit_200": mobilevitv2_200,
    "mobilenet_conv_small": mobilenetv4_conv_small,
    "mobilenet_conv_medium": mobilenetv4_conv_medium,
    "mobilenet_hybrid_medium": mobilenetv4_hybrid_medium,
    "mobilenet_conv_large": mobilenetv4_conv_large,
    "mobilenet_hybrid_large": mobilenetv4_hybrid_large,
}

IN_CHANNELS = 3
NUM_CLASSES = 656


class ModelWrapper(L.LightningModule):
    def __init__(self, arch: str, bs: int = 1):
        super().__init__()
        model_fn = ARCHS[arch]
        self.model = model_fn(num_classes=NUM_CLASSES, in_chans=IN_CHANNELS)
        self.example_input_array = torch.randn(bs, IN_CHANNELS, 224, 224)

    def run(self):
        return self(self.example_input_array)

    def forward(self, x) -> Any:
        return self.model(x)


# headers = ["Model", "#Params", "FLOPs"]
table = []
for model_name_version in tqdm(ARCHS):
    model_name, model_version = model_name_version.split("_", 1)
    with torch.device("cpu"):
        bs = 4 if model_name_version.startswith("mobilenet") else 1
        model = ModelWrapper(model_name_version, bs=bs)

    FLOPs = measure_flops(model, model.run) / bs

    summary = ModelSummary(model=model)

    del model

    table.append(
        [
            f"{model_name}-{model_version}",
            f"{summary.total_parameters / 1e6:.2f} M",
            f"{FLOPs / 1e9:.2f} G",
        ]
    )

100%|██████████| 22/22 [00:04<00:00,  4.41it/s]


In [3]:
df = pd.DataFrame(table, columns=["Model", "#Params", "FLOPs"])
df.head()

,Model,#Params,FLOPs
0,resnet-18,11.51 M,3.63 G
1,resnet-34,21.62 M,7.33 G
2,resnet-50,24.85 M,8.18 G
3,resnet-101,43.84 M,15.60 G
4,resnet-152,59.49 M,23.03 G


In [4]:
df.to_csv("model_info.csv", index=False)